# سيرفر الصوتين الخفيفين على Hugging Face

يسوّي (أو يحدّث) الـSpace `omran77/omran-voices-tts`: عام (الخاص يحتاج حساب مدفوع) بس كل طلب يحتاج مفتاح سري،
والموديلات تبقى في مستودعك الخاص. يرفع ملفاته، ويحط فيه السرّين `HF_TOKEN` و`SPACE_KEY` (مفتاح جديد كل تشغيل)،
وينتظر لين يشتغل، ويقول جملة بكل صوت ويرفعها في `omran77/omran-voices-piper` داخل `samples/` (`space_m.wav`
و`space_f.wav`)، وبعدها يعطي موقع الأصوات العنوان والمفتاح (ملف `tts-space` في مستودعه الخاص). ما تحتاج تضيف شي في Vercel.

**على Kaggle:** Settings ← Accelerator **None** (ما يحتاج كرت) و Internet **On**، و Add-ons ← Secrets ← `HF_TOKEN` (✓)،
ثم Save Version ← **Save & Run All (Commit)**. حوالي ١٥ دقيقة.

In [ ]:
import json, os, secrets, time
from pathlib import Path

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
except ImportError:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
HF_TOKEN = (HF_TOKEN or '').strip()
assert HF_TOKEN.startswith('hf_'), 'السر HF_TOKEN ناقص'
from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
print('Hugging Face account:', api.whoami()['name'])
SPACE = 'omran77/omran-voices-tts'
MODEL_REPO = 'omran77/omran-voices-piper'
SITE_DATASET = 'omran77/omran-voices-recordings'  # private; the voices site reads tts-space from it
KEY = secrets.token_urlsafe(32)
Path('space').mkdir(exist_ok=True)

In [ ]:
%%writefile space/README.md
---
title: Omran Voices TTS
emoji: 🗣️
colorFrom: green
colorTo: blue
sdk: docker
app_port: 7860
pinned: false
---

The two light voices of omran-voices (Piper, formal Arabic): `POST /tts {"text", "voice": "m"|"f"}` → WAV.
Every call needs the owner's key (header `X-Space-Key`); the models live in a private repository.
Source: `space/` in the omran-voices repository.

In [ ]:
%%writefile space/Dockerfile
FROM python:3.11-slim
RUN useradd -m -u 1000 user
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY app.py .
USER user
ENV HF_HOME=/home/user/.cache/huggingface
EXPOSE 7860
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "7860"]

In [ ]:
%%writefile space/requirements.txt
piper-tts==1.8.0
fastapi==0.115.12
uvicorn==0.34.2
huggingface_hub>=0.30,<1

In [ ]:
%%writefile space/app.py
"""The two light voices (Piper, formal Arabic) as a small web service, for a free CPU Space on Hugging Face.

  POST /tts   {"text": "...", "voice": "m" | "f"}  ->  audio/wav (22 050 Hz mono)
  GET  /health                                     ->  {"voices": ["f", "m"], "model": {...}}
  POST /reload                                     ->  loads the newest models (after a training run)
Each needs the header X-Space-Key: the Space's secret SPACE_KEY (without it every call is refused).

The Space itself is public (a private Space needs a paid account: 402 when the notebook made one),
so this code is public and holds no secret. The models stay in the private repo
omran77/omran-voices-piper, read with the Space's secret HF_TOKEN; the key is made by the setup
notebook, which also gives it to the voices site through its private dataset.
MODEL_DIR (a local folder with the same files) replaces the repo for local tests.
"""
import hmac
import io
import os
import threading
import wave
from pathlib import Path

from fastapi import Depends, FastAPI, Header, HTTPException
from fastapi.responses import Response
from pydantic import BaseModel

REPO = os.environ.get('MODEL_REPO', 'omran77/omran-voices-piper')
FILES = {'m': 'ar_AE-omran_m_fusha-medium.onnx', 'f': 'ar_AE-omran_f_fusha-medium.onnx'}
MAX_CHARS = 1200  # the app sends pieces of ~260 characters; this only stops abuse

voices, loaded = {}, {}
lock = threading.Lock()


def fetch(name):
    local = os.environ.get('MODEL_DIR')
    if local:
        return str(Path(local) / name)
    from huggingface_hub import hf_hub_download
    return hf_hub_download(REPO, name, token=os.environ.get('HF_TOKEN') or None, force_download=False)


def load():
    from piper import PiperVoice
    fresh, info = {}, {}
    for spk, name in FILES.items():
        model = fetch(name)
        fetch(name + '.json')
        fresh[spk] = PiperVoice.load(model)  # Arabic: Piper adds the diacritics itself (as in training)
        info[spk] = {'file': name, 'bytes': Path(model).stat().st_size, 'modified': int(Path(model).stat().st_mtime)}
    with lock:
        voices.clear()
        voices.update(fresh)
        loaded.clear()
        loaded.update(info)


app = FastAPI(docs_url=None, redoc_url=None, openapi_url=None)  # nothing to browse on a public Space


def check_key(x_space_key: str = Header(default='')):
    key = os.environ.get('SPACE_KEY', '')
    if not key or not hmac.compare_digest(x_space_key.encode(), key.encode()):
        raise HTTPException(401, 'key')


@app.get('/')
def root():  # Hugging Face's own check that the app is up: says nothing else
    return {'ok': True}


@app.on_event('startup')
def startup():
    load()


class Speak(BaseModel):
    text: str
    voice: str


@app.post('/tts', dependencies=[Depends(check_key)])
def tts(req: Speak):
    text = ' '.join(req.text.split())
    if not text:
        raise HTTPException(400, 'empty text')
    if len(text) > MAX_CHARS:
        raise HTTPException(400, f'text longer than {MAX_CHARS} characters')
    with lock:
        voice = voices.get(req.voice)
    if voice is None:
        raise HTTPException(404, 'unknown voice: ' + req.voice)
    buf = io.BytesIO()
    with wave.open(buf, 'wb') as w:
        voice.synthesize_wav(text, w)
    return Response(buf.getvalue(), media_type='audio/wav', headers={'Cache-Control': 'no-store'})


@app.get('/health', dependencies=[Depends(check_key)])
def health():
    with lock:
        return {'voices': sorted(voices), 'model': dict(loaded)}


@app.post('/reload', dependencies=[Depends(check_key)])
def reload():
    # newest files from the repo (hf_hub_download fetches a file again when the repo has a newer one)
    load()
    return health()

In [ ]:
# the Space (public code, no secret in it), the token it reads the private models with, and its key
api.create_repo(SPACE, repo_type='space', space_sdk='docker', private=False, exist_ok=True)
api.add_space_secret(SPACE, 'HF_TOKEN', HF_TOKEN)
api.add_space_secret(SPACE, 'SPACE_KEY', KEY)
api.upload_folder(folder_path='space', repo_id=SPACE, repo_type='space', commit_message='omran voices tts')
try:  # new secrets are read at start (a Space already building just picks them up)
    api.restart_space(SPACE)
except Exception as e:
    print('restart:', str(e)[:200])
print('uploaded to', SPACE)

In [ ]:
# building the image and loading the two models takes a few minutes
start, stage = time.time(), ''
while time.time() - start < 25 * 60:
    stage = str(api.get_space_runtime(SPACE).stage)
    print(time.strftime('%H:%M:%S'), stage)
    if stage.endswith('RUNNING'):
        break
    if 'ERROR' in stage:
        raise RuntimeError('the Space failed: open https://huggingface.co/spaces/' + SPACE + ' -> Logs')
    time.sleep(30)
assert stage.endswith('RUNNING'), 'the Space did not start in 25 minutes'

In [ ]:
# one sentence in each voice, through the same door the site uses (the key in X-Space-Key)
import requests
url = 'https://' + SPACE.replace('/', '-') + '.hf.space'
auth = {'X-Space-Key': KEY}
for _ in range(20):  # the models may still be loading right after RUNNING
    h = requests.get(url + '/health', headers=auth, timeout=60)
    if h.ok and h.json().get('voices') == ['f', 'm']:
        break
    time.sleep(15)
print('health:', h.status_code, h.text[:300])
print('without the key:', requests.post(url + '/tts', json={'text': 'x', 'voice': 'm'}, timeout=60).status_code, '(must be 401)')
Path('out').mkdir(exist_ok=True)
for spk in ('m', 'f'):
    t = time.time()
    r = requests.post(url + '/tts', headers=auth, timeout=120,
                      json={'text': 'سنلتقي غدا إن شاء الله في البيت بعد صلاة المغرب.', 'voice': spk})
    print(spk, r.status_code, r.headers.get('content-type'), len(r.content), 'bytes in', round(time.time() - t, 1), 's')
    assert r.ok and r.content[:4] == b'RIFF', r.text[:300]
    Path(f'out/space_{spk}.wav').write_bytes(r.content)
api.upload_folder(folder_path='out', path_in_repo='samples', repo_id=MODEL_REPO, commit_message='space test')
print('listen: https://huggingface.co/' + MODEL_REPO + '/tree/main/samples  (space_m.wav, space_f.wav)')
# it works: now the voices site may use it (no extension: the dataset viewer must not read it as data)
api.upload_file(path_or_fileobj=json.dumps({'url': url, 'key': KEY}).encode(), path_in_repo='tts-space',
                repo_id=SITE_DATASET, repo_type='dataset', commit_message='voices server address')
print('the voices site now sends v5 and v4 (Arabic) to', url)